# Soft Sensor Kadar Silika pada Konsentrat Bijih Besi — Proses Flotasi
### *Silica Impurity Soft Sensor for Iron Ore Flotation Process (CRISP-DM)*

**Ringkasan.** Kadar silika (SiO₂) pada konsentrat adalah KPI kualitas utama pabrik flotasi bijih besi,
tetapi baru diketahui setelah analisis laboratorium (per jam, dengan jeda). Proyek ini membangun
*soft sensor* — model yang mengestimasi kadar silika saat ini dari 21 sensor proses dan riwayat hasil
lab — agar operator dapat bertindak lebih cepat (mis. menyesuaikan dosis amina atau aliran udara)
sebelum konsentrat keluar spesifikasi.

| Tahap CRISP-DM | Isi notebook |
|---|---|
| 1. Business Understanding | proses flotasi, masalah jeda lab, tujuan, kriteria sukses |
| 2. Data Understanding | struktur waktu (20 detik vs per jam), kualitas data, EDA, kebocoran data |
| 3. Data Preparation | pembersihan, agregasi per jam, fitur proses & riwayat lab, split berbasis waktu |
| 4. Modeling | demonstrasi *leakage*, model koreksi hasil lab, 3 skenario fitur × 4 model, CV time series, tuning, sensitivitas jeda lab |
| 5. Evaluation | uji pada data terbaru, CI bootstrap, alarm *off-spec*, faktor proses berpengaruh, kesimpulan |

**Cara menjalankan:** letakkan `MiningProcess_Flotation_Plant_Database.csv` di folder yang sama dengan
notebook ini (atau ubah `DATA_PATH` di bawah), lalu *Run All*. Waktu jalan ± 3–6 menit.
Library: `pandas numpy scipy scikit-learn matplotlib` (XGBoost opsional — dipakai otomatis bila terpasang).

In [ ]:
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import HuberRegressor, RidgeCV
from sklearn.metrics import (confusion_matrix, mean_absolute_error, mean_squared_error,
                             precision_score, r2_score, recall_score, roc_auc_score)
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
pd.set_option("display.max_columns", 40)

# ---- gaya visual konsisten (palet ramah buta warna, grid tipis, teks netral)
C = {"blue": "#2a78d6", "orange": "#eb6834", "green": "#1baf7a", "yellow": "#eda100",
     "purple": "#6250d6", "ink": "#0b0b0b", "ink2": "#52514e", "muted": "#898781",
     "grid": "#e1e0d9", "good": "#0ca30c", "bad": "#d03b3b"}
plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#c3c2b7",
    "axes.grid": True, "grid.color": C["grid"], "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.titlepad": 20, "axes.labelcolor": C["ink2"], "xtick.color": C["muted"],
    "ytick.color": C["muted"], "xtick.labelcolor": C["ink2"], "ytick.labelcolor": C["ink2"],
    "legend.frameon": False, "lines.linewidth": 1.8, "font.size": 10,
})


def subtitle(ax, text):
    """Kalimat kesimpulan kecil di bawah judul grafik."""
    ax.text(0, 1.015, text, transform=ax.transAxes, fontsize=9, color=C["ink2"], va="bottom")


def metrics(y, p):
    return {"MAE": mean_absolute_error(y, p), "RMSE": mean_squared_error(y, p) ** 0.5,
            "R2": r2_score(y, p)}


SEED = 42
print("XGBoost tersedia:", HAS_XGB)

## 1. Business Understanding

**Proses.** Pabrik ini memakai *reverse cationic flotation*: bijih (pulp) masuk ke 7 kolom flotasi,
**amina** (kolektor) mengikat partikel silika agar terangkat ke buih (*froth*) dan dibuang, sedangkan
**starch/pati** (depresan) menahan mineral besi agar tetap di bawah sebagai konsentrat. Aliran udara,
level kolom, pH, dan densitas pulp menentukan seberapa efektif pemisahan ini.

**Masalah.** Kadar silika konsentrat diukur di laboratorium setiap jam, dan hasilnya datang terlambat.
Selama menunggu, konsentrat yang keluar spesifikasi terus diproduksi → risiko penalti harga,
*reblending*, atau penolakan produk.

**Tujuan analitik.** Mengestimasi kadar silika jam berjalan (*nowcasting*) dari sensor proses dan hasil
lab yang sudah tersedia, serta memberi **alarm *off-spec*** lebih awal.

**Kriteria sukses (ditetapkan sebelum modeling):**
1. Model lebih akurat (MAE & RMSE) daripada *baseline* operator: "anggap silika sama dengan hasil lab terakhir" (*persistence*).
2. Keunggulan terbukti secara statistik pada data uji periode terbaru (CI bootstrap tidak memuat 0).
3. Alarm *off-spec* (silika > batas) dengan recall ≥ 70% tanpa membanjiri operator dengan alarm palsu.

**Asumsi** (ubah di `CONFIG` bila data perusahaan tersedia):
* Jeda lab = 1 jam → saat memprediksi jam *t*, hasil lab terbaru yang diketahui adalah jam *t − 1*.
* Batas *off-spec* silika = 3,0% (± persentil ke-75 data; spesifikasi kontrak sebenarnya dapat berbeda).

In [ ]:
CONFIG = {
    "DATA_PATH": "MiningProcess_Flotation_Plant_Database.csv",
    "TARGET": "% Silica Concentrate",
    "LEAKY": "% Iron Concentrate",       # hasil lab pada jam yang sama -> tidak boleh jadi fitur
    "LAB_DELAY": 1,                     # jam
    "OFFSPEC": 3.0,                     # % silika
    "TEST_FRACTION": 0.20,              # 20% periode TERAKHIR untuk uji
    "CV_SPLITS": 5,
    "CV_GAP": 24,                       # jam jeda antar train-validasi di CV
}
TARGET, LEAKY = CONFIG["TARGET"], CONFIG["LEAKY"]

## 2. Data Understanding
### 2.1 Memuat data
File memakai **koma sebagai pemisah desimal** (format Brasil) — tanpa `decimal=","` semua angka terbaca sebagai teks.

In [ ]:
t0 = time.time()
path = Path(CONFIG["DATA_PATH"])
if not path.exists():  # cadangan: cari di subfolder data/
    path = Path("data") / path.name
raw = pd.read_csv(path, decimal=",", parse_dates=["date"])
print(f"{raw.shape[0]:,} baris × {raw.shape[1]} kolom | dimuat dalam {time.time() - t0:.1f} detik")
print("Periode:", raw["date"].min(), "→", raw["date"].max())
raw.head()

In [ ]:
groups = {
    "Kualitas umpan (lab)": ["% Iron Feed", "% Silica Feed"],
    "Reagen & pulp": ["Starch Flow", "Amina Flow", "Ore Pulp Flow", "Ore Pulp pH", "Ore Pulp Density"],
    "Aliran udara kolom 1–7": [c for c in raw.columns if "Air Flow" in c],
    "Level kolom 1–7": [c for c in raw.columns if "Level" in c],
    "Kualitas konsentrat (lab)": [LEAKY, TARGET],
}
pd.DataFrame([(g, len(cols), ", ".join(cols[:3]) + (" …" if len(cols) > 3 else ""))
              for g, cols in groups.items()], columns=["Kelompok", "Jumlah", "Contoh kolom"])

In [ ]:
raw.describe().T[["mean", "std", "min", "50%", "max"]]

### 2.2 Struktur waktu & kualitas data
Data sensor tercatat setiap 20 detik, tetapi kolom `date` hanya berisi jam. Mari periksa frekuensi
tiap variabel, celah waktu, dan duplikasi.

In [ ]:
per_hour = raw.groupby("date").size()
full_range = pd.date_range(raw["date"].min(), raw["date"].max(), freq="h")
missing_hours = full_range.difference(per_hour.index)
gaps = pd.Series(per_hour.index).diff()
big_gap = gaps[gaps > pd.Timedelta("1h")]
gap_start = per_hour.index[big_gap.index[0] - 1]
gap_end = per_hour.index[big_gap.index[0]]

target_varies = raw.groupby("date")[TARGET].nunique() > 1
feed_changes = raw.groupby("date")["% Iron Feed"].first().diff().ne(0).sum()

quality = pd.DataFrame([
    ("Baris per jam (median)", int(per_hour.median()), "≈ 180 → sampling 20 detik"),
    ("Baris duplikat persis", int(raw.duplicated().sum()), "dibuang"),
    ("Jam hilang", len(missing_hours), f"celah {gap_start:%d %b} → {gap_end:%d %b %Y}"),
    ("Jam dengan target bervariasi dalam 1 jam", int(target_varies.sum()),
     "nilai lab hasil interpolasi, bukan ukuran asli"),
    ("Perubahan nilai umpan (% Iron Feed)", int(feed_changes),
     f"dari {len(per_hour):,} jam → umpan diukur ±tiap 9 jam"),
    ("Nilai kosong", int(raw.isna().sum().sum()), ""),
], columns=["Pemeriksaan", "Hasil", "Catatan"])
quality

In [ ]:
hourly_target = raw.groupby("date")[TARGET].mean()
fig, axes = plt.subplots(1, 2, figsize=(15, 4.2), gridspec_kw={"width_ratios": [3, 1]})
ax = axes[0]
ax.plot(hourly_target.index, hourly_target.values, color=C["blue"], lw=0.6)
ax.plot(hourly_target.rolling(24 * 7, min_periods=24).mean(), color=C["orange"], lw=2,
        label="Rata-rata bergerak 7 hari")
ax.axvspan(gap_start, gap_end, color=C["muted"], alpha=0.15, lw=0)
ax.text(gap_start, 5.3, "  celah data\n  13 hari", fontsize=8, color=C["ink2"], va="top")
ax.axhline(CONFIG["OFFSPEC"], color=C["bad"], ls="--", lw=1)
ax.text(hourly_target.index[-1], CONFIG["OFFSPEC"] + 0.08, f"batas off-spec {CONFIG['OFFSPEC']}%",
        ha="right", fontsize=8, color=C["ink2"])
ax.set_ylabel("% silika konsentrat")
ax.legend(loc="upper right", fontsize=8)
ax.set_title("Kadar silika konsentrat per jam (Maret–September 2017)")
subtitle(ax, "Sangat fluktuatif jam ke jam; level rata-rata bergeser antar bulan (proses tidak stasioner)")
ax = axes[1]
ax.hist(hourly_target, bins=40, color=C["blue"], alpha=0.85, orientation="horizontal")
ax.axhline(CONFIG["OFFSPEC"], color=C["bad"], ls="--", lw=1)
ax.set_xlabel("Jumlah jam")
ax.set_title("Distribusi")
subtitle(ax, f"{(hourly_target > CONFIG['OFFSPEC']).mean():.0%} jam di atas batas")
plt.tight_layout()
plt.show()

### 2.3 Hubungan variabel dengan kadar silika
Korelasi dihitung pada rata-rata per jam (satuan waktu target yang sebenarnya).

In [ ]:
hourly_mean = raw.groupby("date").mean()
corr = hourly_mean.corr()[TARGET].drop(TARGET).sort_values()
fig, ax = plt.subplots(figsize=(9, 7))
colors = [C["bad"] if c == LEAKY else (C["orange"] if v > 0 else C["blue"]) for c, v in corr.items()]
ax.barh(corr.index, corr.values, color=colors, height=0.65)
ax.axvline(0, color=C["muted"], lw=1)
ax.set_xlabel("Korelasi Pearson dengan % silika konsentrat (data per jam)")
ax.set_title("Korelasi variabel terhadap kadar silika")
subtitle(ax, "% Iron Concentrate (merah) = hasil lab pada saat yang sama → KEBOCORAN DATA; sensor proses hanya |r| ≤ 0,22")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
ax = axes[0]
lags = range(1, 25)
ac = [hourly_target.autocorr(l) for l in lags]
ax.bar(list(lags), ac, color=C["blue"], width=0.7)
ax.set_xlabel("Lag (jam)"); ax.set_ylabel("Autokorelasi")
ax.set_title("Autokorelasi kadar silika")
subtitle(ax, f"Silika 1 jam lalu berkorelasi {ac[0]:.2f} dengan sekarang → riwayat lab sangat informatif")
ax.grid(axis="x", visible=False)
ax = axes[1]
s = hourly_mean.sample(1500, random_state=SEED)
ax.scatter(s[LEAKY], s[TARGET], s=8, alpha=0.4, color=C["blue"], lw=0)
ax.set_xlabel("% Iron Concentrate"); ax.set_ylabel("% Silica Concentrate")
ax.set_title("Mengapa % Iron Concentrate dikeluarkan")
subtitle(ax, "Hampir kebalikan langsung dari target; diukur di lab yang sama, pada jam yang sama")
plt.tight_layout()
plt.show()

In [ ]:
key_vars = ["Amina Flow", "Starch Flow", "Ore Pulp pH", "Ore Pulp Density",
            "Flotation Column 01 Air Flow", "% Silica Feed"]
fig, axes = plt.subplots(2, 3, figsize=(15, 7.5))
for ax, col in zip(axes.ravel(), key_vars):
    bins = pd.qcut(hourly_mean[col], 10, duplicates="drop")
    g = hourly_mean.groupby(bins, observed=True)[TARGET].agg(["mean", "std", "count"])
    x = hourly_mean.groupby(bins, observed=True)[col].mean()
    ci = 1.96 * g["std"] / np.sqrt(g["count"])
    ax.errorbar(x, g["mean"], yerr=ci, fmt="o-", color=C["blue"], ecolor=C["ink2"],
                elinewidth=1, capsize=3, ms=5)
    ax.set_xlabel(col); ax.set_ylabel("Rata-rata % silika")
    ax.set_title(col, fontsize=11)
    subtitle(ax, f"r = {hourly_mean[[col, TARGET]].corr().iloc[0, 1]:+.2f}")
fig.suptitle("Kadar silika per desil variabel proses (95% CI)", x=0.01, ha="left",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

**Ringkasan Data Understanding**
* **Tiga frekuensi berbeda dalam satu tabel:** sensor proses tiap 20 detik, hasil lab konsentrat per jam,
  kualitas umpan ±tiap 9 jam. Nilai lab "diulang" ±180 kali per jam.
* **Kualitas data:** 1.171 baris duplikat, celah 13 hari (16–29 Maret), 310 jam dengan target hasil interpolasi.
* **Kebocoran data:** `% Iron Concentrate` adalah hasil lab yang sama waktunya dengan target (r = −0,80) → dibuang.
* **Sinyal:** hubungan sensor proses dengan silika lemah (|r| ≤ 0,22) dan arahnya masuk akal secara proses
  (aliran udara & level lebih tinggi → silika turun; amina lebih tinggi → silika lebih tinggi, kemungkinan
  respons operator terhadap kualitas buruk). Sebaliknya, riwayat silika sangat informatif (autokorelasi 0,78).

## 3. Data Preparation
Keputusan:
1. **Buang duplikat** dan **gunakan data setelah celah 13 hari** (29 Maret 12:00 ke atas) agar deret waktu kontinu.
2. **Agregasi per jam** — satuan target yang sebenarnya. Untuk tiap sensor: rata-rata (*level* proses) dan
   standar deviasi (stabilitas proses dalam jam tersebut).
3. **Jam dengan target interpolasi** tidak dipakai sebagai target latih/uji (bukan ukuran lab asli),
   tetapi tetap dipakai untuk menghitung riwayat.
4. **Fitur proses:** rata-rata aliran udara & level 7 kolom, dosis reagen per aliran pulp (starch/pulp,
   amina/pulp), rasio silika/besi umpan, rata-rata bergerak 3 jam (waktu tinggal material di sirkuit).
5. **Fitur riwayat lab:** lag silika yang **benar-benar sudah tersedia** sesuai jeda lab (t−1, t−2, t−3, t−6,
   t−12, t−24) dan rata-rata 6 jam terakhir.
6. **Split berbasis waktu:** 80% periode awal untuk latih, 20% periode terakhir untuk uji. Tidak boleh acak.

In [ ]:
df = raw.drop_duplicates()
process_cols = [c for c in df.columns if c not in ["date", TARGET, LEAKY]]

hourly = df.groupby("date")[process_cols].agg(["mean", "std"])
hourly.columns = [f"{c}__{s}" for c, s in hourly.columns]
hourly = hourly.drop(columns=["% Iron Feed__std", "% Silica Feed__std"])  # umpan konstan per jam
hourly[TARGET] = df.groupby("date")[TARGET].mean()
hourly["target_interpolated"] = df.groupby("date")[TARGET].nunique() > 1
hourly = hourly.asfreq("h")
hourly = hourly[hourly.index >= gap_end]
print(f"Data per jam: {hourly.shape[0]:,} jam | jam kosong setelah celah: {hourly[TARGET].isna().sum()}")


def build_features(h: pd.DataFrame, lab_delay: int):
    """Mengembalikan (DataFrame fitur+target, kolom sensor, kolom riwayat lab)."""
    h = h.copy()
    air = [c for c in h if "Air Flow__mean" in c]
    lvl = [c for c in h if "Level__mean" in c]
    h["air_flow_avg"] = h[air].mean(axis=1)
    h["column_level_avg"] = h[lvl].mean(axis=1)
    h["starch_dosage"] = h["Starch Flow__mean"] / h["Ore Pulp Flow__mean"]
    h["amina_dosage"] = h["Amina Flow__mean"] / h["Ore Pulp Flow__mean"]
    h["feed_silica_iron_ratio"] = h["% Silica Feed__mean"] / h["% Iron Feed__mean"]
    for c in ["Amina Flow__mean", "Starch Flow__mean", "Ore Pulp pH__mean",
              "Ore Pulp Density__mean", "air_flow_avg", "column_level_avg"]:
        h[f"{c}_roll3h"] = h[c].rolling(3, min_periods=1).mean()
    sensor_cols = [c for c in h if c not in [TARGET, "target_interpolated"]]

    for lag in (1, 2, 3, 6, 12, 24):
        h[f"silica_lag{lag + lab_delay - 1}h"] = h[TARGET].shift(lag + lab_delay - 1)
    h["silica_mean_last6h"] = h[TARGET].shift(lab_delay).rolling(6).mean()
    h["silica_std_last6h"] = h[TARGET].shift(lab_delay).rolling(6).std()
    lab_cols = [c for c in h if c.startswith("silica_")]

    h = h.dropna()
    h = h[~h["target_interpolated"].astype(bool)]
    return h, sensor_cols, lab_cols


data, SENSOR_COLS, LAB_COLS = build_features(hourly, CONFIG["LAB_DELAY"])
cut = data.index[int(len(data) * (1 - CONFIG["TEST_FRACTION"]))]
train, test = data[data.index < cut], data[data.index >= cut]
PERSIST_COL = f"silica_lag{CONFIG['LAB_DELAY']}h"

print(f"Fitur sensor: {len(SENSOR_COLS)} | fitur riwayat lab: {len(LAB_COLS)}")
print(f"Train: {train.index.min():%d %b} → {train.index.max():%d %b %Y} ({len(train):,} jam)")
print(f"Test : {test.index.min():%d %b} → {test.index.max():%d %b %Y} ({len(test):,} jam)")
print(f"Proporsi off-spec — train {(train[TARGET] > CONFIG['OFFSPEC']).mean():.1%}, "
      f"test {(test[TARGET] > CONFIG['OFFSPEC']).mean():.1%}")

## 4. Modeling
### 4.1 Jebakan umum: split acak pada data 20 detik
Banyak analisis pada dataset ini melatih model langsung pada baris 20 detik dengan `train_test_split` acak.
Karena nilai target **sama** untuk ±180 baris dalam satu jam, baris "uji" hampir selalu punya kembaran
di data latih → skor tampak sangat tinggi tetapi tidak berlaku saat model dipakai pada jam-jam berikutnya.

In [ ]:
sub = df[df["date"] >= gap_end].sample(120_000, random_state=SEED)
Xa, Xb, ya, yb = train_test_split(sub[process_cols], sub[TARGET], test_size=0.2, random_state=SEED)
rf_demo = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=SEED).fit(Xa, ya)
r2_random = r2_score(yb, rf_demo.predict(Xb))

rf_time = RandomForestRegressor(n_estimators=300, min_samples_leaf=10, max_features=0.3,
                                n_jobs=-1, random_state=SEED).fit(train[SENSOR_COLS], train[TARGET])
r2_time = r2_score(test[TARGET], rf_time.predict(test[SENSOR_COLS]))

leak_demo = pd.DataFrame({
    "Desain evaluasi": ["Split acak, baris 20 detik (cara umum)", "Split waktu, data per jam (realistis)"],
    "Fitur": ["sensor proses", "sensor proses"],
    "R² uji": [r2_random, r2_time]})
leak_demo

R² ±0,85 dari split acak adalah **ilusi**. Dengan evaluasi yang meniru pemakaian nyata (melatih di masa lalu,
menguji di masa depan), sensor proses saja hanya menjelaskan ±5% variasi silika. Seluruh evaluasi
selanjutnya memakai **split berbasis waktu** dan **TimeSeriesSplit** (5 fold, jeda 24 jam).

### 4.2 Formulasi model
Dua keputusan desain berbasis domain:
1. **Prediksi koreksi, bukan nilai absolut.** Untuk skenario yang memakai riwayat lab, model memprediksi
   *perubahan* silika terhadap hasil lab terakhir: `silika_t = lab_terakhir + koreksi`. Ini cara berpikir
   operator ("dari hasil lab terakhir, kira-kira naik atau turun?") dan membuat model tidak perlu
   mempelajari ulang level rata-rata yang terus bergeser antar bulan.
2. **Loss sesuai metrik bisnis.** Error absolut (MAE, dalam % silika) mudah dipahami operator. Model dengan
   *squared loss* (Ridge) cenderung kalah MAE dari persistence, sehingga dibandingkan juga **Huber** dan
   **Gradient Boosting dengan absolute loss**.

Skenario fitur:
* **A — sensor saja:** *soft sensor* murni (tanpa lab).
* **B — riwayat lab saja:** model dinamika kualitas.
* **C — sensor + riwayat lab:** pendekatan *hybrid*.

In [ ]:
class LastLabCorrection(BaseEstimator, RegressorMixin):
    """Prediksi = hasil lab terakhir yang tersedia + koreksi dari model dasar."""

    def __init__(self, estimator, features, lag_col):
        self.estimator = estimator
        self.features = features
        self.lag_col = lag_col

    def fit(self, X, y):
        self.est_ = clone(self.estimator).fit(X[self.features], np.asarray(y) - X[self.lag_col].to_numpy())
        return self

    def predict(self, X):
        return X[self.lag_col].to_numpy() + self.est_.predict(X[self.features])


def base_models():
    zoo = {
        "Ridge": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 4, 20))),
        "Huber": make_pipeline(StandardScaler(), HuberRegressor(alpha=1.0, max_iter=1000)),
        "GBM (absolute loss)": HistGradientBoostingRegressor(
            loss="absolute_error", learning_rate=0.03, max_iter=300, max_leaf_nodes=15,
            min_samples_leaf=40, random_state=SEED),
        "Random Forest": RandomForestRegressor(n_estimators=200, min_samples_leaf=10, max_features=0.3,
                                               n_jobs=-1, random_state=SEED),
    }
    if HAS_XGB:
        zoo["XGBoost (absolute loss)"] = XGBRegressor(
            objective="reg:absoluteerror", n_estimators=400, learning_rate=0.03, max_depth=4,
            subsample=0.8, colsample_bytree=0.7, random_state=SEED, n_jobs=-1)
    return zoo


def make_model(scenario, base, sensor_cols, lab_cols, lag_col):
    if scenario.startswith("A"):
        return _Direct(clone(base), sensor_cols)
    feats = lab_cols if scenario.startswith("B") else sensor_cols + lab_cols
    return LastLabCorrection(clone(base), feats, lag_col)


class _Direct(BaseEstimator, RegressorMixin):
    """Model langsung pada subset kolom (dipakai skenario A)."""

    def __init__(self, estimator, features):
        self.estimator = estimator
        self.features = features

    def fit(self, X, y):
        self.est_ = clone(self.estimator).fit(X[self.features], y)
        return self

    def predict(self, X):
        return self.est_.predict(X[self.features])


SCENARIOS = ["A. Sensor saja", "B. Riwayat lab saja", "C. Sensor + riwayat lab"]
tscv = TimeSeriesSplit(n_splits=CONFIG["CV_SPLITS"], gap=CONFIG["CV_GAP"])


def cv_evaluate(model, d):
    scores = []
    for i_tr, i_va in tscv.split(d):
        tr, va = d.iloc[i_tr], d.iloc[i_va]
        scores.append(metrics(va[TARGET], clone(model).fit(tr, tr[TARGET]).predict(va)))
    return pd.DataFrame(scores).mean()

### 4.3 Perbandingan skenario × model (cross-validation time series, hanya data latih)

In [ ]:
rows = []
for i_tr, i_va in tscv.split(train):
    tr, va = train.iloc[i_tr], train.iloc[i_va]
    rows.append({"skenario": "Baseline", "model": "Rata-rata", **metrics(va[TARGET], np.full(len(va), tr[TARGET].mean()))})
    rows.append({"skenario": "Baseline", "model": "Persistence", **metrics(va[TARGET], va[PERSIST_COL])})
cv_rows = pd.DataFrame(rows).groupby(["skenario", "model"]).mean().reset_index().to_dict("records")

for scen in SCENARIOS:
    for name, base in base_models().items():
        m = make_model(scen, base, SENSOR_COLS, LAB_COLS, PERSIST_COL)
        cv_rows.append({"skenario": scen, "model": name, **cv_evaluate(m, train)})
cv_summary = pd.DataFrame(cv_rows).sort_values("MAE").reset_index(drop=True)
persist_cv = cv_summary.query("model == 'Persistence'").iloc[0]
cv_summary["MAE vs persistence"] = cv_summary["MAE"] / persist_cv["MAE"] - 1
cv_summary["RMSE vs persistence"] = cv_summary["RMSE"] / persist_cv["RMSE"] - 1
cv_summary

In [ ]:
plot_df = cv_summary.query("skenario != 'Baseline'")
scen_colors = {"A. Sensor saja": C["muted"], "B. Riwayat lab saja": C["blue"],
               "C. Sensor + riwayat lab": C["orange"]}
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2), sharey=True)
labels = [f"{r.skenario} · {r.model}" for r in plot_df.itertuples()]
for ax, metric in zip(axes, ["MAE", "RMSE"]):
    ax.barh(labels, plot_df[metric], color=[scen_colors[s] for s in plot_df["skenario"]], height=0.62)
    ax.axvline(persist_cv[metric], color=C["ink"], ls="--", lw=1.2)
    ax.text(persist_cv[metric], len(plot_df) - 0.75, f" persistence = {persist_cv[metric]:.3f}",
            fontsize=8.5, color=C["ink"], bbox={"facecolor": "white", "edgecolor": "none", "pad": 1.5})
    ax.set_xlabel(f"{metric} cross-validation (% silika) — lebih kecil lebih baik")
    ax.set_title(metric, fontsize=11)
    ax.grid(axis="y", visible=False)
axes[0].invert_yaxis()
axes[0].set_xlim(0.35, None); axes[1].set_xlim(0.6, None)
subtitle(axes[0], "Biru = riwayat lab, oranye = sensor + lab, abu = sensor saja")
subtitle(axes[1], "Garis putus = baseline persistence (hasil lab terakhir)")
fig.suptitle("Perbandingan skenario fitur × model (TimeSeriesSplit 5 fold, gap 24 jam)",
             x=0.01, ha="left", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

**Bacaan hasil:**
* Sensor saja (abu) tidak lebih baik dari menebak rata-rata → hubungan sensor–silika lemah dan bergeser.
* Model koreksi lab dengan **absolute loss** mengalahkan persistence pada MAE dan RMSE.
* Menambahkan sensor (oranye) **tidak** memperbaiki model berbasis lab — temuan penting untuk
  rekomendasi instrumentasi di bagian 5.

### 4.4 Tuning model terbaik
Ruang pencarian sengaja kecil dan berregularisasi kuat: data hanya ±2.900 jam dan proses tidak stasioner.

In [ ]:
best_row = cv_summary.query("skenario != 'Baseline'").iloc[0]
BEST_SCEN = best_row["skenario"]
print(f"Kandidat terbaik sebelum tuning: {BEST_SCEN} · {best_row['model']} | CV MAE {best_row['MAE']:.4f}")

space = {
    "estimator__learning_rate": [0.01, 0.02, 0.03, 0.05],
    "estimator__max_iter": [150, 300, 500],
    "estimator__max_leaf_nodes": [4, 7, 15],
    "estimator__min_samples_leaf": [20, 40, 80, 160],
    "estimator__l2_regularization": [0.0, 1.0, 10.0],
}
gbm_base = base_models()["GBM (absolute loss)"]
search = RandomizedSearchCV(make_model(BEST_SCEN, gbm_base, SENSOR_COLS, LAB_COLS, PERSIST_COL),
                            space, n_iter=20, cv=tscv, scoring="neg_mean_absolute_error",
                            random_state=SEED, n_jobs=-1)
search.fit(train, train[TARGET])
tuned_cv = cv_evaluate(search.best_estimator_, train)
print(f"GBM tuned ({BEST_SCEN}) CV MAE = {tuned_cv['MAE']:.4f} | RMSE = {tuned_cv['RMSE']:.4f}")
print("Parameter terbaik:", {k.replace('estimator__', ''): v for k, v in search.best_params_.items()})

if tuned_cv["MAE"] < best_row["MAE"]:
    final_model, FINAL_NAME = search.best_estimator_, f"{BEST_SCEN} · GBM absolute loss (tuned)"
else:
    final_model = make_model(BEST_SCEN, base_models()[best_row["model"]], SENSOR_COLS, LAB_COLS, PERSIST_COL)
    FINAL_NAME = f"{BEST_SCEN} · {best_row['model']}"
print("MODEL FINAL:", FINAL_NAME)

### 4.5 Seberapa berharga model bila hasil lab makin terlambat?
Di lapangan jeda lab bisa 1–4 jam (antrean sampel, pergantian shift, gangguan alat). Semakin lama jeda,
semakin "basi" hasil lab terakhir. Konfigurasi model final dilatih ulang untuk setiap skenario jeda (CV data latih).

In [ ]:
delay_rows = []
for delay in [1, 2, 3, 4]:
    d, s_cols, l_cols = build_features(hourly, delay)
    d = d[d.index < cut]
    lag_col = f"silica_lag{delay}h"
    m = make_model(BEST_SCEN, final_model.estimator, s_cols, l_cols, lag_col)
    res = cv_evaluate(m, d)
    pers = pd.DataFrame([metrics(d.iloc[v][TARGET], d.iloc[v][lag_col]) for _, v in tscv.split(d)]).mean()
    delay_rows.append({"jeda_lab_jam": delay, "MAE persistence": pers["MAE"], "MAE model": res["MAE"],
                       "RMSE persistence": pers["RMSE"], "RMSE model": res["RMSE"]})
delay_df = pd.DataFrame(delay_rows).set_index("jeda_lab_jam")
delay_df["perbaikan MAE"] = 1 - delay_df["MAE model"] / delay_df["MAE persistence"]
delay_df["perbaikan RMSE"] = 1 - delay_df["RMSE model"] / delay_df["RMSE persistence"]
display(delay_df)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
for ax, metric in zip(axes, ["MAE", "RMSE"]):
    ax.plot(delay_df.index, delay_df[f"{metric} persistence"], marker="o", color=C["muted"],
            label="Persistence (lab terakhir)")
    ax.plot(delay_df.index, delay_df[f"{metric} model"], marker="o", color=C["blue"], label="Model final")
    for x, r in delay_df.iterrows():
        ax.annotate(f"{-r[f'perbaikan {metric}']:+.0%}", (x, r[f"{metric} model"]), xytext=(10, -12),
                    textcoords="offset points", ha="left", fontsize=9, color=C["ink"])
    ax.set_xticks(delay_df.index)
    ax.set_xlabel("Jeda hasil lab (jam)"); ax.set_ylabel(f"{metric} CV (% silika)")
    ax.set_title(f"{metric} vs jeda laboratorium", fontsize=11)
    ax.legend(loc="upper left", fontsize=8.5)
subtitle(axes[0], "Angka = perubahan error model dibanding persistence")
subtitle(axes[1], "Keunggulan model membesar saat hasil lab makin terlambat")
plt.tight_layout()
plt.show()

## 5. Evaluation
### 5.1 Uji pada periode terbaru (test set yang belum pernah dilihat)

In [ ]:
final_model.fit(train, train[TARGET])
pred_test = pd.Series(final_model.predict(test), index=test.index)
persist_test = test[PERSIST_COL]

hybrid = make_model("C. Sensor + riwayat lab", final_model.estimator, SENSOR_COLS, LAB_COLS, PERSIST_COL)
hybrid.fit(train, train[TARGET])

results = pd.DataFrame({
    "Rata-rata data latih": metrics(test[TARGET], np.full(len(test), train[TARGET].mean())),
    "Persistence (lab terakhir)": metrics(test[TARGET], persist_test),
    "Sensor saja (RF)": metrics(test[TARGET], rf_time.predict(test[SENSOR_COLS])),
    "Sensor + riwayat lab (konfigurasi sama)": metrics(test[TARGET], hybrid.predict(test)),
    f"FINAL — {FINAL_NAME}": metrics(test[TARGET], pred_test),
}).T
for m in ["MAE", "RMSE"]:
    results[f"{m} vs persistence"] = results[m] / results.loc["Persistence (lab terakhir)", m] - 1
results

In [ ]:
# Apakah keunggulan atas persistence nyata? Bootstrap per BLOK HARIAN (menjaga autokorelasi dalam satu hari)
err_model, err_pers = (test[TARGET] - pred_test).abs(), (test[TARGET] - persist_test).abs()
days = test.index.normalize()
day_idx = {d: np.flatnonzero(days == d) for d in days.unique()}
rng = np.random.default_rng(SEED)
boot_mae, boot_rmse = [], []
for _ in range(2000):
    idx = np.concatenate([day_idx[d] for d in rng.choice(list(day_idx), len(day_idx), replace=True)])
    em, ep = err_model.iloc[idx], err_pers.iloc[idx]
    boot_mae.append(ep.mean() - em.mean())
    boot_rmse.append(np.sqrt((ep ** 2).mean()) - np.sqrt((em ** 2).mean()))
ci = pd.DataFrame({
    "selisih (persistence − model)": [err_pers.mean() - err_model.mean(),
                                      np.sqrt((err_pers ** 2).mean()) - np.sqrt((err_model ** 2).mean())],
    "CI 95% bawah": [np.percentile(boot_mae, 2.5), np.percentile(boot_rmse, 2.5)],
    "CI 95% atas": [np.percentile(boot_mae, 97.5), np.percentile(boot_rmse, 97.5)],
}, index=["MAE", "RMSE"])
ci["model lebih baik secara signifikan?"] = ci["CI 95% bawah"] > 0
ci

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={"width_ratios": [2.2, 1]})
ax = axes[0]
# reindex per jam agar jam yang dibuang (target interpolasi) tampil sebagai celah, bukan garis lurus
window = pd.date_range(test.index[-1] - pd.Timedelta(days=7), test.index[-1], freq="h")
ax.plot(window, test[TARGET].reindex(window), color=C["ink"], lw=1.4, label="Aktual (lab)")
ax.plot(window, persist_test.reindex(window), color=C["muted"], lw=1.2, ls="--", label="Persistence (lab terakhir)")
ax.plot(window, pred_test.reindex(window), color=C["blue"], lw=1.8, label="Model final")
ax.set_ylim(0.5, 6.3)
ax.axhline(CONFIG["OFFSPEC"], color=C["bad"], lw=1, ls=":")
ax.set_ylabel("% silika")
ax.legend(loc="upper left", fontsize=8, ncol=3)
ax.set_title("7 hari terakhir data uji")
subtitle(ax, "Persistence selalu tertinggal 1 jam; model mengoreksi sebagian lonjakan")
ax = axes[1]
ax.scatter(test[TARGET], pred_test, s=8, alpha=0.4, color=C["blue"], lw=0)
lim = [test[TARGET].min() - 0.1, test[TARGET].max() + 0.1]
ax.plot(lim, lim, color=C["muted"], ls="--", lw=1)
ax.set_xlabel("Aktual (% silika)"); ax.set_ylabel("Prediksi model")
ax.set_title("Prediksi vs aktual (seluruh data uji)")
subtitle(ax, f"R² = {r2_score(test[TARGET], pred_test):.2f} (persistence {r2_score(test[TARGET], persist_test):.2f})")
plt.tight_layout()
plt.show()

In [ ]:
daily = pd.DataFrame({"Persistence": err_pers, "Model": err_model}).groupby(days).mean()
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(daily.index, daily["Persistence"], color=C["muted"], marker="o", ms=3, label="Persistence")
ax.plot(daily.index, daily["Model"], color=C["blue"], marker="o", ms=3, label="Model final")
ax.set_ylabel("MAE harian (% silika)")
ax.legend(loc="upper left", fontsize=8.5)
ax.set_title("Stabilitas error per hari pada periode uji")
subtitle(ax, f"Model lebih baik pada {(daily['Model'] < daily['Persistence']).mean():.0%} hari")
plt.tight_layout()
plt.show()

### 5.2 Alarm *off-spec* untuk operator
Alarm berbunyi bila estimasi silika melewati ambang. Ambang **dipilih dari prediksi out-of-fold data latih**
(bukan dari data uji): recall ≥ 70% dengan precision setinggi mungkin.

In [ ]:
oof_final = pd.Series(index=train.index, dtype=float)
for i_tr, i_va in tscv.split(train):
    tr, va = train.iloc[i_tr], train.iloc[i_va]
    oof_final.iloc[i_va] = clone(final_model).fit(tr, tr[TARGET]).predict(va)
mask = oof_final.notna()
y_oof = (train.loc[mask, TARGET] > CONFIG["OFFSPEC"]).astype(int)

grid = np.round(np.arange(2.0, 3.6, 0.05), 2)
cand = [(t, recall_score(y_oof, oof_final[mask] > t), precision_score(y_oof, oof_final[mask] > t, zero_division=0))
        for t in grid]
feasible = [c for c in cand if c[1] >= 0.70]
ALARM_T = max(feasible, key=lambda c: c[2])[0] if feasible else CONFIG["OFFSPEC"]
print(f"Ambang alarm terpilih (dari OOF data latih): estimasi silika > {ALARM_T}%")

y_true = (test[TARGET] > CONFIG["OFFSPEC"]).astype(int)
alarm_rows = []
for name, score, thr in (("Persistence (lab terakhir > batas)", persist_test, CONFIG["OFFSPEC"]),
                         ("Persistence dengan ambang yang sama", persist_test, ALARM_T),
                         (f"Model final (estimasi > {ALARM_T}%)", pred_test, ALARM_T)):
    flag = (score > thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, flag).ravel()
    alarm_rows.append({"metode": name, "recall": tp / (tp + fn), "precision": tp / max(tp + fp, 1),
                       "jam off-spec tertangkap": f"{tp}/{tp + fn}", "alarm palsu": int(fp),
                       "ROC-AUC": roc_auc_score(y_true, score)})
alarm = pd.DataFrame(alarm_rows)
alarm

### 5.3 Apakah sensor proses membawa informasi? (permutation importance model *hybrid* pada data uji)
Meskipun sensor tidak menambah akurasi total, engineer tetap perlu tahu variabel proses mana yang
paling "didengar" model — sebagai bahan diskusi kontrol proses.

In [ ]:
# Importance diukur pada model KOREKSI (target = silika − lab terakhir), agar efek struktural
# "lab terakhir sebagai titik awal" tidak mendominasi.
inner, feats = hybrid.est_, hybrid.features
y_corr = test[TARGET] - test[PERSIST_COL]
pi = permutation_importance(inner, test[feats], y_corr, n_repeats=10, scoring="neg_mean_absolute_error",
                            random_state=SEED, n_jobs=1)
imp = (pd.DataFrame({"fitur": feats, "kenaikan_MAE": pi.importances_mean, "std": pi.importances_std})
         .sort_values("kenaikan_MAE", ascending=False).head(15))
share_sensor = imp.loc[~imp["fitur"].str.startswith("silica_"), "kenaikan_MAE"].clip(lower=0).sum() / \
    imp["kenaikan_MAE"].clip(lower=0).sum()
fig, ax = plt.subplots(figsize=(9, 5.5))
is_lab = imp["fitur"].str.startswith("silica_")
ax.barh(imp["fitur"], imp["kenaikan_MAE"], xerr=imp["std"],
        color=[C["blue"] if l else C["orange"] for l in is_lab], height=0.6,
        error_kw={"ecolor": C["ink2"], "lw": 1})
ax.axvline(0, color=C["muted"], lw=1)
ax.invert_yaxis()
ax.set_xlabel("Kenaikan MAE koreksi bila fitur diacak (% silika)")
ax.set_title("15 fitur terpenting untuk mengoreksi hasil lab terakhir — model hybrid")
subtitle(ax, f"Biru = riwayat lab, oranye = sensor proses; sensor menyumbang ±{share_sensor:.0%} dari importance top-15")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

### 5.4 Evaluasi terhadap kriteria sukses

In [ ]:
fin = results.iloc[-1]
pers = results.loc["Persistence (lab terakhir)"]
model_alarm = alarm.iloc[-1]
criteria = pd.DataFrame([
    ("Lebih akurat dari persistence — MAE", f"{pers['MAE']:.3f} → {fin['MAE']:.3f} ({fin['MAE vs persistence']:+.1%})",
     "LOLOS" if fin["MAE"] < pers["MAE"] else "GAGAL"),
    ("Lebih akurat dari persistence — RMSE", f"{pers['RMSE']:.3f} → {fin['RMSE']:.3f} ({fin['RMSE vs persistence']:+.1%})",
     "LOLOS" if fin["RMSE"] < pers["RMSE"] else "GAGAL"),
    ("Keunggulan MAE signifikan (CI 95% > 0)", f"[{ci.loc['MAE', 'CI 95% bawah']:.3f}, {ci.loc['MAE', 'CI 95% atas']:.3f}]",
     "LOLOS" if ci.loc["MAE", "CI 95% bawah"] > 0 else "GAGAL"),
    ("Keunggulan RMSE signifikan (CI 95% > 0)", f"[{ci.loc['RMSE', 'CI 95% bawah']:.3f}, {ci.loc['RMSE', 'CI 95% atas']:.3f}]",
     "LOLOS" if ci.loc["RMSE", "CI 95% bawah"] > 0 else "GAGAL"),
    ("Alarm off-spec recall ≥ 70%", f"{model_alarm['recall']:.0%} (precision {model_alarm['precision']:.0%})",
     "LOLOS" if model_alarm["recall"] >= 0.70 else "GAGAL"),
], columns=["Kriteria", "Hasil", "Status"])
print(f"Total waktu notebook: {(time.time() - t0) / 60:.1f} menit")
criteria

### 5.5 Kesimpulan & rekomendasi
**Temuan utama** (angka dari eksekusi notebook; dapat sedikit berbeda antar versi library)
1. **Evaluasi yang jujur mengubah kesimpulan.** Split acak pada data 20 detik memberi R² ≈ 0,86, tetapi itu
   kebocoran data (target sama ±180 kali per jam). Dengan split berbasis waktu, sensor proses saja hanya
   menjelaskan ≈ 6% variasi silika — sensor belum bisa menggantikan laboratorium.
2. **Model koreksi hasil lab (GBM, absolute loss) mengalahkan kebiasaan "pakai hasil lab terakhir"** pada
   periode uji terbaru: MAE −3,8% dan RMSE −5,0%, keduanya signifikan (CI 95% bootstrap harian > 0).
3. **Nilai model membesar saat lab terlambat:** pada jeda 4 jam error turun ±10% (MAE) dan ±14% (RMSE)
   dibanding persistence — relevan saat shift sibuk, antrean sampel, atau gangguan alat lab.
4. **Sensor proses tidak menambah akurasi** di atas riwayat lab pada validasi berbasis waktu. Hubungan
   sensor–silika lemah dan bergeser antar bulan; ini temuan instrumentasi, bukan kegagalan model.
5. **Alarm off-spec** (estimasi > ambang dari data latih) menangkap ±71% jam off-spec dengan precision ±56%.
   Kemampuan membedakan jam off-spec (ROC-AUC ≈ 0,87) setara dengan hasil lab terakhir; keunggulan model
   ada pada estimasi angka yang lebih akurat, bukan pada alarmnya.

**Rekomendasi operasional**
* Gunakan model sebagai **estimasi antara** di antara dua hasil lab dan sebagai pendamping alarm, bukan pengganti lab.
* Catat **waktu aktual** hasil lab keluar → manfaat model terbesar saat jeda panjang.
* **Retraining berkala** (mis. bulanan) dan pantau MAE harian, karena karakter proses bergeser.

**Rekomendasi instrumentasi & data** (agar soft sensor berbasis sensor benar-benar bekerja)
* Ukuran partikel umpan (P80) dan mineralogi, setpoint reagen beserta waktu perubahannya,
  kamera buih (*froth vision*: kecepatan & warna buih), serta *on-stream analyzer* bila memungkinkan.
* Sinkronisasi waktu sampel lab dengan data sensor (saat ini hanya tercatat per jam).

**Keterbatasan:** satu pabrik dan ±5 bulan data; batas off-spec 3,0% dan jeda lab 1 jam adalah asumsi;
data periode uji (Agustus–September) mungkin berbeda karakter dengan periode lain.